In [47]:
import pandas as pd

df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")
composers = pd.read_csv("composers.csv")

df_train.head()

,id,pitch,duration
0,25,41,0.79
1,25,51,0.45
2,25,20,0.57
3,25,51,0.69
4,25,62,0.34


In [48]:
df_train.id.nunique()

400

In [49]:
composers = composers.sort_values(by='datapointID')
#composers

In [50]:
def song_to_text(song_df):
    docs = []

    for song_id in song_df["id"].unique():
        notes = song_df[song_df["id"] == song_id].sort_index()

        tokens = [
            f"{row.pitch}_{row.duration:.2f}"
            for row in notes.itertuples()
        ]

        docs.append(" ".join(tokens))

    return docs

new_df = pd.DataFrame()
new_df["id"] = df_train["id"].unique()
new_df["songs"] = song_to_text(df_train)
new_df = new_df.sort_values(by='id')
new_df["composer"] = composers["answer"]
df_train = new_df

df_train.head()

,id,songs,composer
364,0,73_0.11 53_0.45 59_0.80 23_0.23 49_0.89 21_0.7...,The Beatles
345,1,29_0.58 47_0.77 63_0.14 23_0.93 67_0.79 78_0.9...,David Bowie
21,2,29_0.76 43_0.77 35_0.63 62_0.77 87_0.54 71_0.7...,David Bowie
304,4,34_0.66 57_0.73 37_0.26 44_1.00 76_0.83 27_0.9...,David Bowie
175,5,27_0.30 43_0.26 48_0.22 69_0.77 60_0.63 69_0.3...,Led Zeppelin


In [51]:
#df_test = get_feats(df_test)
#df_test.head()

In [54]:
target = 'composer'

X = df_train['songs']
y = df_train[target]

#X_final = df_test[features]

from sklearn.feature_extraction.text import TfidfVectorizer
vect = TfidfVectorizer(ngram_range=(1, 4))
X = vect.fit_transform(X)

X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 710954 stored elements and shape (400, 419661)>

In [56]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import cross_val_score

model = LogisticRegression()

cross_val_score(model, X, y, cv=3, scoring='f1_weighted').mean()

np.float64(0.13906383105275777)